Runs as is on rapidsai/notebooks:24.10a-cuda12.5-py3.10 

In [ ]:
%pip install polars hdbscan azure-storage-blob dtale

In [ ]:
import cuml
import cupy as cp
from cuml.cluster.hdbscan import HDBSCAN

In [ ]:
%env AZURE_STORAGE_ACCOUNT_KEY=

In [ ]:
import numpy as np
from azure.storage.blob import BlobServiceClient
import io
import os

# Replace with your actual connection string
account_name="enclaveid"
account_key=os.environ["AZURE_STORAGE_ACCOUNT_KEY"]
connection_string = f"DefaultEndpointsProtocol=https;AccountName={account_name};AccountKey={account_key};EndpointSuffix=core.windows.net"
container_name = 'enclaveid-dagster-prod-bucket'
blob_name = 'interests_embeddings/cm0i27jdj0000aqpa73ghpcxf.snappy'

# Create the BlobServiceClient object
blob_service_client = BlobServiceClient.from_connection_string(connection_string)

# Create the BlobClient object
blob_client = blob_service_client.get_blob_client(container=container_name, blob=blob_name)

# Download the blob content to a bytes buffer
downloaded_blob = blob_client.download_blob()

# Save the blob data to a file
with open("data.snappy", "wb") as file:
    downloaded_blob.readinto(file)

import polars as pl
df = pl.read_parquet("data.snappy")

In [ ]:
import numpy as np

def get_cluster_centroids(embeddings_gpu, cluster_labels: np.ndarray):
    unique_labels = np.unique(cluster_labels)
    cluster_centroids = {}
    for label in unique_labels:
        if label != -1:  # Skip noise points
            cluster_embeddings = embeddings_gpu[cluster_labels == label]
            cluster_centroid = cp.mean(cluster_embeddings, axis=0)
            cluster_centroids[label] = cluster_centroid
    return cluster_centroids

def get_cluster_stats(cluster_labels: np.ndarray, prefix=""):
    cluster_stats = np.unique(cluster_labels, return_counts=True)
    return {
        f"{prefix}clusters_count": len(cluster_stats[0]),
        f"{prefix}noise_count": int(cluster_stats[1][0])
        if -1 in cluster_stats[0]
        else 0,
    }

In [ ]:
# Convert the embeddings to a CuPy array
embeddings_gpu = cp.asarray(df["embeddings"].to_numpy())

# Reduce the embeddings dimensions
umap_model = cuml.UMAP(
    n_neighbors=15, n_components=100, min_dist=0.1, metric="cosine"
)
reduced_data_gpu = umap_model.fit_transform(embeddings_gpu)

# Clustering for single interests
fine_cluster_labels = HDBSCAN(
    min_cluster_size=5,
    gen_min_span_tree=True,
    metric="euclidean",
).fit_predict(reduced_data_gpu.astype(np.float64).get())


# Calculate centroids of fine clusters
fine_cluster_centroids = get_cluster_centroids(
    reduced_data_gpu.get(), fine_cluster_labels
)

coarse_cluster_labels = HDBSCAN(
    min_cluster_size=2,
    cluster_selection_epsilon=0.15,
    gen_min_span_tree=True,
    metric="euclidean",
).fit_predict(np.array(list(fine_cluster_centroids.values())))

coarse_cluster_mapping = {
    old_label: new_label
    for old_label, new_label in zip(
        fine_cluster_centroids.keys(), coarse_cluster_labels
    )
}
merged_cluster_labels = np.array(
    [coarse_cluster_mapping.get(label, -1) for label in fine_cluster_labels]
)

result = df.with_columns(
    cluster_label=pl.Series(fine_cluster_labels),
    merged_cluster_label=pl.Series(merged_cluster_labels),
).rename({"embeddings": "interests_embeddings"})

In [ ]:
get_cluster_stats(merged_cluster_labels, prefix="coarse_")

In [ ]:
get_cluster_stats(fine_cluster_labels, prefix="fine_")

In [ ]:
from dtale import show
import dtale.global_state as global_state

global_state.set_app_settings(dict(max_column_width=300))
d = show(df.to_pandas())
#d.open_browser()

In [ ]:
result.write_parquet("result.snappy")